In [ ]:
import os, json, shutil, subprocess, tempfile
from pathlib import Path
from Bio.PDB import PDBParser

# ---- paths ----
FOLDSEEK = "foldseek"
PDB_DIR  = Path("../../../data/rBMRB/cluster_30p/structures_split_filtered/test")
OUT_JSON = Path("../../../data/analysis/Fig4/tdi_results/foldseek_3di_per_residue.json")

pdb_files = sorted(PDB_DIR.glob("*.pdb"))
print(f"Found {len(pdb_files)} PDB files")
assert shutil.which(FOLDSEEK), "foldseek not found on PATH"

In [ ]:
# ---- 1) run Foldseek once on all PDBs to get AA + 3Di sequences ----
# structureto3didescriptor writes one line per structure:  name <tab> AA_seq <tab> 3Di_seq
# symlink the .pdb files into a temporary folder so other files do not confuse foldseek
tmp = Path(tempfile.mkdtemp())
for f in pdb_files:
    os.symlink(f.resolve(), tmp / f.name)   # absolute target: PDB_DIR is relative

tsv = tmp / "all_3di.tsv"
res = subprocess.run([FOLDSEEK, "structureto3didescriptor", str(tmp) + "/", str(tsv)],
                     capture_output=True, text=True)
print("foldseek exit code:", res.returncode)
if res.returncode != 0:
    print(res.stderr[-2000:])
    raise RuntimeError("foldseek failed")

# parse TSV  ->  {stem: (aa_seq, tdi_seq)}
fs = {}
with open(tsv) as fh:
    for line in fh:
        parts = line.rstrip("\n").split("\t")
        if len(parts) < 3:
            continue
        name, aa, tdi = parts[0], parts[1], parts[2]
        stem = name[:-4] if name.endswith(".pdb") else name  # 10008_1.pdb -> 10008_1
        fs[stem] = (aa, tdi)
print(f"Parsed 3Di for {len(fs)} structures")

In [ ]:
# ---- 2) read the real resnum of every Calpha residue and align it with 3Di ----
# Foldseek follows the residue order of the PDB file, as does Bio.PDB iteration over Calpha atoms.
parser = PDBParser(QUIET=True)

def ca_residues(pdb_path):
    """Return the ordered (resnum, resname_3letter) list of residues that have a Calpha."""
    structure = parser.get_structure("x", str(pdb_path))
    out = []
    for residue in structure[0].get_residues():
        if "CA" in residue:
            out.append((residue.id[1], residue.resname))
    return out

result = {}
mismatch = []  # structures whose lengths do not match

for f in pdb_files:
    stem = f.stem
    if stem not in fs:
        mismatch.append((stem, "no foldseek output"))
        continue
    aa, tdi = fs[stem]
    cas = ca_residues(f)
    if not (len(aa) == len(tdi) == len(cas)):
        mismatch.append((stem, f"len aa={len(aa)} tdi={len(tdi)} ca={len(cas)}"))
        continue
    records = []
    for (resnum, _resname3), aa_char, tdi_char in zip(cas, aa, tdi):
        records.append({"resnum": resnum, "resname": aa_char, "tdi": tdi_char})
    result[stem] = records

print(f"Aligned & wrote records for {len(result)} structures")
if mismatch:
    print(f"⚠️  {len(mismatch)} structures skipped (length mismatch):")
    for s, why in mismatch[:20]:
        print("   ", s, why)

In [ ]:
# ---- 3) save JSON ----
OUT_JSON = "../../../data/analysis/Fig4/tdi_results/foldseek_3di_per_residue.json"
with open(OUT_JSON, "w") as fh:
    json.dump(result, fh, indent=2)
shutil.rmtree(tmp, ignore_errors=True)
print("Saved:", OUT_JSON)
print("num structures:", len(result))